<a href="https://colab.research.google.com/github/banulaperera/SDGP_PT_Y3_03_Intellectia/blob/IN-29-model_creation/model/Intellectia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

In [ ]:
!pip install -U tensorflow-text==2.15

In [3]:
# Importing libraries

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from matplotlib import pyplot as plt
import seaborn as sn
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

In [4]:
preprocess_url = hub.load('https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3')
encoder_url = hub.load('https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4')

In [5]:
bert_preprocess = hub.KerasLayer(preprocess_url)
bert_encoder = hub.KerasLayer(encoder_url)

In [6]:
# Import dataset

dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/arXiv-DataFrame.csv')
dataset.head()

,Unnamed: 0,id,Title,Summary,Author,Link,Publish Date,Update Date,Primary Category,Category
0,0,cs/9308101v1,Dynamic Backtracking,Because of their occasional need to return to ...,M. L. Ginsberg,http://arxiv.org/pdf/cs/9308101v1,1993-08-01T00:00:00Z,1993-08-01T00:00:00Z,cs.AI,['cs.AI']
1,1,cs/9308102v1,A Market-Oriented Programming Environment and ...,Market price systems constitute a well-underst...,M. P. Wellman,http://arxiv.org/pdf/cs/9308102v1,1993-08-01T00:00:00Z,1993-08-01T00:00:00Z,cs.AI,['cs.AI']
2,2,cs/9309101v1,An Empirical Analysis of Search in GSAT,We describe an extensive study of search in GS...,I. P. Gent,http://arxiv.org/pdf/cs/9309101v1,1993-09-01T00:00:00Z,1993-09-01T00:00:00Z,cs.AI,['cs.AI']
3,3,cs/9311101v1,The Difficulties of Learning Logic Programs wi...,As real logic programmers normally use cut (!)...,F. Bergadano,http://arxiv.org/pdf/cs/9311101v1,1993-11-01T00:00:00Z,1993-11-01T00:00:00Z,cs.AI,['cs.AI']
4,4,cs/9311102v1,Software Agents: Completing Patterns and Const...,To support the goal of allowing users to recor...,J. C. Schlimmer,http://arxiv.org/pdf/cs/9311102v1,1993-11-01T00:00:00Z,1993-11-01T00:00:00Z,cs.AI,['cs.AI']


In [ ]:
dataset

,Unnamed: 0,id,Title,Summary,Author,Link,Publish Date,Update Date,Primary Category,Category
0,0,cs/9308101v1,Dynamic Backtracking,Because of their occasional need to return to ...,M. L. Ginsberg,http://arxiv.org/pdf/cs/9308101v1,1993-08-01T00:00:00Z,1993-08-01T00:00:00Z,cs.AI,['cs.AI']
1,1,cs/9308102v1,A Market-Oriented Programming Environment and ...,Market price systems constitute a well-underst...,M. P. Wellman,http://arxiv.org/pdf/cs/9308102v1,1993-08-01T00:00:00Z,1993-08-01T00:00:00Z,cs.AI,['cs.AI']
2,2,cs/9309101v1,An Empirical Analysis of Search in GSAT,We describe an extensive study of search in GS...,I. P. Gent,http://arxiv.org/pdf/cs/9309101v1,1993-09-01T00:00:00Z,1993-09-01T00:00:00Z,cs.AI,['cs.AI']
3,3,cs/9311101v1,The Difficulties of Learning Logic Programs wi...,As real logic programmers normally use cut (!)...,F. Bergadano,http://arxiv.org/pdf/cs/9311101v1,1993-11-01T00:00:00Z,1993-11-01T00:00:00Z,cs.AI,['cs.AI']
4,4,cs/9311102v1,Software Agents: Completing Patterns and Const...,To support the goal of allowing users to recor...,J. C. Schlimmer,http://arxiv.org/pdf/cs/9311102v1,1993-11-01T00:00:00Z,1993-11-01T00:00:00Z,cs.AI,['cs.AI']
...,...,...,...,...,...,...,...,...,...,...
53469,53469,math/0603084v1,Advances on nonparametric regression for funct...,We consider the problem of predicting a real r...,Frédéric Ferraty,http://arxiv.org/pdf/math/0603084v1,2006-03-03T13:25:42Z,2006-03-03T13:25:42Z,math.ST,"['math.ST', 'stat.TH']"
53470,53470,math/0603123v1,Ranking and empirical minimization of U-statis...,"The problem of ranking/ordering instances, ins...",Stéphan Clémençon,http://arxiv.org/pdf/math/0603123v1,2006-03-05T17:10:54Z,2006-03-05T17:10:54Z,math.ST,"['math.ST', 'stat.TH', '68Q32, 60G99, 62G99, 6..."
53471,53471,math/0603130v1,Nonparametric methods for inference in the pre...,"We suggest two nonparametric approaches, based...",Peter Hall,http://arxiv.org/pdf/math/0603130v1,2006-03-06T07:31:27Z,2006-03-06T07:31:27Z,math.ST,"['math.ST', 'stat.TH', '62G08 (Primary) 62G20 ..."
53472,53472,math/0603132v1,Functional linear regression analysis for long...,We propose nonparametric methods for functiona...,Fang Yao,http://arxiv.org/pdf/math/0603132v1,2006-03-06T08:09:42Z,2006-03-06T08:09:42Z,math.ST,"['math.ST', 'stat.TH', '62M20 (Primary) 60G15,..."


In [ ]:
dataset.isnull().sum()

Unnamed: 0          0
id                  0
Title               0
Summary             0
Author              0
Link                0
Publish Date        0
Update Date         0
Primary Category    0
Category            0
dtype: int64

In [ ]:
primaryCategoryBalance = dataset['Primary Category'].value_counts()
print(primaryCategoryBalance.to_frame().to_string())

                    Primary Category
math.ST                          702
math-ph                          701
cs.IT                            700
cond-mat.stat-mech               383
cond-mat.soft                    373
econ.GN                          371
physics.bio-ph                   369
cs.CY                            368
q-fin.GN                         367
q-bio.BM                         364
cs.DC                            363
q-bio.QM                         362
q-bio.MN                         362
cs.SE                            361
cs.AI                            360
physics.soc-ph                   358
cs.NI                            358
cs.MA                            357
cs.CR                            357
cs.SY                            356
math.HO                          356
cs.PF                            356
cond-mat.dis-nn                  355
cs.DL                            355
quant-ph                         355
cs.DB                            355
c

In [7]:
df = dataset[["Summary", "Primary Category"]]
df

,Summary,Primary Category
0,Because of their occasional need to return to ...,cs.AI
1,Market price systems constitute a well-underst...,cs.AI
2,We describe an extensive study of search in GS...,cs.AI
3,As real logic programmers normally use cut (!)...,cs.AI
4,To support the goal of allowing users to recor...,cs.AI
...,...,...
53469,We consider the problem of predicting a real r...,math.ST
53470,"The problem of ranking/ordering instances, ins...",math.ST
53471,"We suggest two nonparametric approaches, based...",math.ST
53472,We propose nonparametric methods for functiona...,math.ST


In [ ]:
df.groupby('Primary Category').describe()

Summary         \
                   count unique   
Primary Category                  
astro-ph.CO          350    350   
astro-ph.EP          350    350   
astro-ph.GA          350    350   
astro-ph.HE          350    350   
astro-ph.IM          350    350   
...                  ...    ...   
stat.AP              350    341   
stat.CO              350    350   
stat.ME              352    327   
stat.ML              351    351   
stat.OT              349    349   

                                                                          
                                                                top freq  
Primary Category                                                          
astro-ph.CO       We consider Brans-Dicke type nonminimally coup...    1  
astro-ph.EP       We report on the discovery of HAT-P-11b, the s...    1  
astro-ph.GA       The magnetic fields of our Milky Way galaxy ar...    1  
astro-ph.HE       We have analyzed 866 RXTE observations of the ...    1  
astro-ph.IM       The use of conventional neutrino telescope met...    1  
...                                                             ...  ...  
stat.AP           Discussion of ``Statistical analysis of an arc...    7  
stat.CO           The problem of the definition and the estimati...    1  
stat.ME           Discussion of ``The William Kruskal Legacy: 19...    7  
stat.ML           Over the past decade, the stellar growth of In...    1  
stat.OT           Statistics is running the risk of appearing ir...    1  

[154 rows x 4 columns]

In [ ]:
df['Primary Category'].value_counts()

math.ST               702
math-ph               701
cs.IT                 700
cond-mat.stat-mech    383
cond-mat.soft         373
                     ... 
q-fin.ST              123
comp-gas              114
cs.GL                 104
physics.atom-ph       100
cond-mat                4
Name: Primary Category, Length: 154, dtype: int64

In [8]:
dummies = pd.get_dummies(df['Primary Category'])

In [9]:
dummies

,astro-ph.CO,astro-ph.EP,astro-ph.GA,astro-ph.HE,astro-ph.IM,astro-ph.SR,comp-gas,cond-mat,cond-mat.dis-nn,cond-mat.mes-hall,...,q-fin.PR,q-fin.RM,q-fin.ST,q-fin.TR,quant-ph,stat.AP,stat.CO,stat.ME,stat.ML,stat.OT
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53469,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53470,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53471,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53472,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
merged = pd.concat([df, dummies], axis = 'columns')
merged

,Summary,Primary Category,astro-ph.CO,astro-ph.EP,astro-ph.GA,astro-ph.HE,astro-ph.IM,astro-ph.SR,comp-gas,cond-mat,...,q-fin.PR,q-fin.RM,q-fin.ST,q-fin.TR,quant-ph,stat.AP,stat.CO,stat.ME,stat.ML,stat.OT
0,Because of their occasional need to return to ...,cs.AI,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Market price systems constitute a well-underst...,cs.AI,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,We describe an extensive study of search in GS...,cs.AI,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,As real logic programmers normally use cut (!)...,cs.AI,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,To support the goal of allowing users to recor...,cs.AI,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53469,We consider the problem of predicting a real r...,math.ST,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53470,"The problem of ranking/ordering instances, ins...",math.ST,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53471,"We suggest two nonparametric approaches, based...",math.ST,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53472,We propose nonparametric methods for functiona...,math.ST,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
le = LabelEncoder()
test = le.fit_transform(df['Primary Category'])

In [12]:
df['y'] = pd.Series(test)

<ipython-input-12-26944894c812>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['y'] = pd.Series(test)


In [13]:
df

,Summary,Primary Category,y
0,Because of their occasional need to return to ...,cs.AI,17
1,Market price systems constitute a well-underst...,cs.AI,17
2,We describe an extensive study of search in GS...,cs.AI,17
3,As real logic programmers normally use cut (!)...,cs.AI,17
4,To support the goal of allowing users to recor...,cs.AI,17
...,...,...,...
53469,We consider the problem of predicting a real r...,math.ST,99
53470,"The problem of ranking/ordering instances, ins...",math.ST,99
53471,"We suggest two nonparametric approaches, based...",math.ST,99
53472,We propose nonparametric methods for functiona...,math.ST,99


In [ ]:
preprocessed = bert_preprocess(df['Summary'])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(df['Summary'], dummies, stratify = df['y'])

In [21]:
X_train

26683    We introduce the notion of scattering space $S...
9387     Fast Fourier Transform (FFT) is an efficient a...
23127    The X-ray numbers of some classes of convex bo...
29220    We propose a nonlinear self-consistent model o...
39820    In the context of the recently developed "equa...
                               ...                        
18909    For $p_1,...,p_n>0$, let $\mathbb E=\{z\in\mat...
39238    A kink-based path integral method, previously ...
35896    We investigate the properties of localized wav...
33094    Several prototypes of a Cherenkov Correlated T...
19413    The functional equation f(p(z))=g(q(z)) is stu...
Name: Summary, Length: 40105, dtype: object

In [22]:
y_train

,astro-ph.CO,astro-ph.EP,astro-ph.GA,astro-ph.HE,astro-ph.IM,astro-ph.SR,comp-gas,cond-mat,cond-mat.dis-nn,cond-mat.mes-hall,...,q-fin.PR,q-fin.RM,q-fin.ST,q-fin.TR,quant-ph,stat.AP,stat.CO,stat.ME,stat.ML,stat.OT
26683,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9387,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23127,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29220,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39820,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18909,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39238,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
35896,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33094,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess(text_input)
outputs = bert_encoder(preprocessed_text)

# Neural network layers
l = tf.keras.layers.Dropout(0.1, name="dropout")(outputs['pooled_output'])
l = tf.keras.layers.Dense(154, activation='softmax', name="output")(l)

# Use inputs and outputs to construct a final model
model = tf.keras.Model(inputs=[text_input], outputs = [l])

In [24]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 keras_layer (KerasLayer)    {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_word_ids': (None,                                            
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             }                                                                

In [25]:
METRICS = [
      tf.keras.metrics.CategoricalAccuracy('accuracy')
]

model.compile(optimizer='adam',
              loss='CategoricalCrossentropy',
              metrics=METRICS)

In [26]:
y_train.shape

(40105, 154)

In [27]:
X_train.shape

(40105,)

In [ ]:
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
1254/1254 [==============================] - 448s 347ms/step - loss: 4.5637 - accuracy: 0.0539
Epoch 2/10
1254/1254 [==============================] - 434s 346ms/step - loss: 3.9442 - accuracy: 0.1210
Epoch 3/10
1254/1254 [==============================] - 434s 346ms/step - loss: 3.6155 - accuracy: 0.1652
Epoch 4/10
1254/1254 [==============================] - 434s 346ms/step - loss: 3.4106 - accuracy: 0.1964
Epoch 5/10
1254/1254 [==============================] - 433s 345ms/step - loss: 3.2631 - accuracy: 0.2183
Epoch 6/10
1254/1254 [==============================] - 433s 346ms/step - loss: 3.1492 - accuracy: 0.2370
Epoch 7/10
1254/1254 [==============================] - 433s 345ms/step - loss: 3.0664 - accuracy: 0.2543
Epoch 8/10
1254/1254 [==============================] - 433s 345ms/step - loss: 2.9926 - accuracy: 0.2654
Epoch 9/10
  84/1254 [=>............................] - ETA: 6:43 - loss: 2.9350 - accuracy: 0.2723

In [ ]:
model.evaluate(X_test, y_test)

418/418 [==============================] - 145s 344ms/step - loss: 3.7392 - accuracy: 0.1560


[3.739164352416992, 0.15603260695934296]

In [ ]:
predicted = model.predict(["Understanding the W boson as accurately as possible, including knowing its mass, has been a priority in particle physics for decades. In the past few years, in a succession of increasing-precision measurements by multiple experiments, a significant tension between the measured and predicted mass has been documented by the CDF Collaboration. Furthermore, smaller differences between different measurements exist. Because the W boson mass provides a window on new physics, a comparison between different measurement techniques can inform the path to further investigations. "])

1/1 [==============================] - 0s 154ms/step


In [ ]:
predicted

In [ ]:
list(le.classes_)[np.argmax(predicted)]

'physics.ed-ph'

In [ ]:
df['Primary Category'][0]

'cs.AI'